# 👨‍🍳 SounderPy Beginner's Cookbook

**A beginner-friendly collection of recipes for retrieving, analyzing, plotting, and exporting atmospheric vertical-profile data.**

**Updated for SounderPy 3.2**

This notebook is designed for users who are new to SounderPy, new to atmospheric
sounding analysis in Python, or simply want a quick tour of the package.

**Recipes in this cookbook**

1. **Set up SounderPy**
2. **Retrieve your first observed sounding**
3. **Understand `clean_data`**
4. **Customize a SounderPy sounding**
5. **Build and modify a hodograph**
6. **Retrieve other data sources**
7. **Calculate sounding parameters**
8. **Compare multiple profiles**
9. **Export a profile**
10. **Use the SounderPy command line**

**Estimated time:** about 20–30 minutes, depending on network speed.

> **Note:** Several recipes retrieve live or archived data from external services.
> An internet connection is required, and occasional upstream archive outages are
> outside SounderPy's control.

*(C) Kyle J. Gillett, University of North Dakota, 2024–2026)*

## Before We Get Started

You do **not** need to be an experienced Python programmer to use this notebook.
A few terms will appear repeatedly:

- **Function / tool** — reusable code that performs a task, such as retrieving
  data or building a sounding.
- **Argument** — information passed into a function.
- **Keyword argument (`kwarg`)** — an optional named setting passed to a function.
- **Dictionary** — a Python object that stores values using named keys.
- **Alias** — a shorthand name used when importing a package. We will use `spy`
  as the alias for SounderPy.
- **`clean_data`** — SounderPy's common vertical-profile data structure.

### Helpful resources

- [SounderPy documentation](https://kylejgillett.github.io/sounderpy/)
- [SounderPy tutorials](https://kylejgillett.github.io/sounderpy/tutorials/index.html)
- [SounderPy plot gallery](https://kylejgillett.github.io/sounderpy/examplegallery.html)
- [Unidata Python Training](https://unidata.github.io/python-training/)

If you are running this notebook in Google Colab and SounderPy is not already
installed, run the installation cell below.

In [ ]:
# Uncomment this line if SounderPy is not installed in your environment.
# %pip install -q sounderpy


**## Recipe 1 — Set Up SounderPy**

**Goal**: Import SounderPy and make its public tools available in this notebook.

The conventional alias is `spy`:

In [ ]:
import sounderpy as spy


That's all the setup we need.

Throughout the cookbook, calls such as:

```python
spy.get_obs_data(...)
spy.build_sounding(...)
spy.build_hodograph(...)
```

are using functions provided by SounderPy.

**## Recipe 2 — Retrieve Your First Observed Sounding**

**Goal**: Retrieve a real observed radiosonde profile and create a full SounderPy sounding.

We will use the Omaha, Nebraska (`OAX`) sounding from **16 June 2014 at 18 UTC**,
during the Pilger, Nebraska severe-weather event.

SounderPy retrieves observed radiosonde data with:

```python
spy.get_obs_data(station, year, month, day, hour)
```

In [ ]:
obs_data = spy.get_obs_data(
    "OAX",
    "2014", "06", "16", "18",
    hush=True,
)


Now build the sounding:

In [ ]:
spy.build_sounding(
    obs_data,
    radar=None,
    map_zoom=0,
)


### 🧠 What you learned

- `get_obs_data()` retrieves an observed atmospheric profile.
- The returned object can be passed directly into `build_sounding()`.
- `radar=None` disables the radar inset.
- `map_zoom=0` hides the map, which keeps this first example simple and avoids
  unnecessary external radar/map requests.

This **retrieve → plot** pattern is the core SounderPy workflow.

**## Recipe 3 — Understand `clean_data`**

**Goal**: Understand what SounderPy actually returned.

SounderPy converts supported observations, forecasts, reanalyses, aircraft
profiles, and model data into a common dictionary often referred to as
`clean_data`.

Inspect the available keys:

In [ ]:
obs_data.keys()


The core profile variables are typically:

| Key | Meaning |
|---|---|
| `p` | pressure |
| `z` | height |
| `T` | temperature |
| `Td` | dewpoint |
| `u` | u-component wind |
| `v` | v-component wind |
| `site_info` | profile metadata |

The meteorological arrays retain physical units through Pint/MetPy.

Let's inspect the metadata:

In [ ]:
obs_data["site_info"]


And look at a few values from the profile:

In [ ]:
print("Pressure:", obs_data["p"][:5])
print("Temperature:", obs_data["T"][:5])
print("Dewpoint:", obs_data["Td"][:5])
print("U wind:", obs_data["u"][:5])
print("V wind:", obs_data["v"][:5])


### Why `clean_data` matters

Once a data source has been converted into this common structure, the same
SounderPy plotting, calculation, and export functions can be used regardless of
where the profile came from.

Conceptually:

```text
RAOB ─────┐
RAP/RUC ──┤
BUFKIT ───┤
ACARS ────┼──> clean_data ──> calculations
WRF ──────┤                  ├─> sounding
CM1 ──────┤                  ├─> hodograph
custom ───┘                  └─> export
```

**## Recipe 4 — Customize a SounderPy Sounding**

**Goal**: Learn a few of the most useful `build_sounding()` options.

The default plot contains a large amount of information, but its appearance and
layout can be customized.

## A simplified parcel display

`special_parcels="simple"` keeps the standard SB/ML/MU CAPE parcel traces while
omitting the additional MU ECAPE parcel trace.

In [ ]:
spy.build_sounding(
    obs_data,
    special_parcels="simple",
    radar=None,
    map_zoom=0,
)


## Dark mode and color-deficiency-friendly colors

`dark_mode=True` changes the presentation theme.

`color_blind=True` changes the conventional red/green thermodynamic trace
combination to a more color-deficiency-friendly presentation.

In [ ]:
spy.build_sounding(
    obs_data,
    special_parcels="simple",
    radar=None,
    map_zoom=0,
    dark_mode=True,
    color_blind=True,
)


## Show potential-temperature traces

SounderPy can also display potential-temperature information with
`show_theta=True`:

In [ ]:
spy.build_sounding(
    obs_data,
    special_parcels="simple",
    radar=None,
    map_zoom=0,
    show_theta=True,
)


### 🍳 Try it yourself

Experiment with one option at a time:

```python
spy.build_sounding(obs_data, dark_mode=True)
spy.build_sounding(obs_data, color_blind=True)
spy.build_sounding(obs_data, map_zoom=4)
```

For archived cases, radar availability depends on the selected radar mode and
the underlying archive. If you are learning the plotting API, `radar=None` is
the most reliable starting point.

**## Recipe 5 — Build and Modify a Hodograph**

**Goal**: Create both ground-relative and storm-relative hodographs.

A standard hodograph is created with:

In [ ]:
spy.build_hodograph(
    obs_data,
    radar=None,
    map_zoom=0,
)


## Storm-relative hodograph

With `sr_hodo=True`, SounderPy subtracts the selected storm-motion vector from
the wind profile.

Here we use the canonical right-moving storm motion:

In [ ]:
spy.build_hodograph(
    obs_data,
    storm_motion="right_moving",
    sr_hodo=True,
    radar=None,
    map_zoom=0,
)


SounderPy also accepts a custom storm motion as:

```python
[direction_degrees, speed_knots]
```

For example:

In [ ]:
spy.build_hodograph(
    obs_data,
    storm_motion=[250.0, 45.0],
    radar=None,
    map_zoom=0,
)


### 🧠 What you learned

- `build_hodograph()` accepts the same `clean_data` object as `build_sounding()`.
- `sr_hodo=False` gives a ground-relative hodograph.
- `sr_hodo=True` gives a storm-relative hodograph.
- Storm motion can be calculated by SounderPy or supplied by the user.

**## Recipe 6 — Retrieve Other Data Sources**

**Goal**: See how the same SounderPy workflow applies to forecasts, model analyses, and aircraft observations.

You do **not** need to run every example to complete the cookbook.


## A. RAP/RUC analysis

For RAP/RUC data, use the model key `"rap-ruc"`.

This example retrieves a RAP analysis valid at **18 UTC 28 August 2024** near
central South Dakota:

In [ ]:
rap_data = spy.get_model_data(
    "rap-ruc",
    [44.58, -100.82],
    "2024", "08", "28", "18",
    box_avg_size=0.25,
    hush=True,
)


In [ ]:
spy.build_sounding(
    rap_data,
    special_parcels="simple",
    radar=None,
    map_zoom=0,
)


> **Note:** RAP/RUC retrieval can take longer than a radiosonde request because
> archived GRIB2 data must be located, downloaded, parsed, spatially sampled,
> and cleaned before the profile is returned.


## B. BUFKIT forecast

BUFKIT provides archived model forecast profiles.

This example retrieves a GFS forecast from the **12 UTC 5 August 2023** run at
`KMOP`, forecast hour 6:

In [ ]:
bufkit_data = spy.get_bufkit_data(
    "gfs",
    "KMOP",
    6,
    "2023", "08", "05", "12",
    hush=True,
)


In [ ]:
spy.build_sounding(
    bufkit_data,
    special_parcels="simple",
    radar=None,
    map_zoom=0,
)


## C. ACARS aircraft observations

ACARS profiles use a two-step workflow:

1. list the profiles available for a date/hour;
2. retrieve one profile by ID.

In [ ]:
acars = spy.acars_data(
    "2024", "05", "21", "18"
)

profiles = acars.list_profiles()
profiles[:10]


In [ ]:
# Retrieve the first available profile from the returned list.
# You can replace profiles[0] with another profile ID.
acars_data = acars.get_profile(
    profiles[0],
    hush=True,
)


### 🧠 The important pattern

Despite coming from very different sources, the returned objects can all be used
with the same downstream functions:

```python
spy.build_sounding(rap_data)
spy.build_sounding(bufkit_data)
spy.build_sounding(acars_data)
```

That common workflow is one of SounderPy's main design goals.

**## Recipe 7 — Calculate Sounding Parameters**

**Goal**: Access SounderPy's calculated thermodynamic and kinematic parameters directly, without relying only on the values printed on a figure.

Use `sounding_params()`:

In [ ]:
general, thermo, kinem, intrp = spy.sounding_params(
    obs_data,
    storm_motion="right_moving",
).calc()


The calculation returns four dictionaries:

- `general` — general profile quantities;
- `thermo` — thermodynamic parameters;
- `kinem` — kinematic parameters;
- `intrp` — interpolated profile data used by many calculations.

Let's inspect a few common severe-weather parameters:

In [ ]:
print("MLCAPE:", thermo["mlcape"])
print("MUCAPE:", thermo["mucape"])
print("MU ECAPE:", thermo["mu_ecape"])

print("0–1 km SRH:", kinem["srh_0_to_1000"])
print("0–3 km SRH:", kinem["srh_0_to_3000"])
print("0–6 km shear:", kinem["shear_0_to_6000"])
print("Effective-layer STP:", kinem["eil_stp"])


### 🍳 Try it yourself

Explore the available keys:

```python
thermo.keys()
kinem.keys()
general.keys()
```

This is often the easiest way to discover which calculated fields are available
for further analysis.

**## Recipe 8 — Compare Multiple Profiles**

**Goal**: Use `build_composite()` to compare the evolution of profiles through time.

Instead of comparing unrelated locations, we'll retrieve three observations from
the same station around the Pilger event:

- OAX — 12 UTC 16 June 2014
- OAX — 18 UTC 16 June 2014
- OAX — 00 UTC 17 June 2014

In [ ]:
oax_12z = spy.get_obs_data(
    "OAX", "2014", "06", "16", "12", hush=True
)

oax_18z = obs_data

oax_00z = spy.get_obs_data(
    "OAX", "2014", "06", "17", "00", hush=True
)


In [ ]:
data_list = [
    oax_12z,
    oax_18z,
    oax_00z,
]

spy.build_composite(
    data_list,
    shade_between=False,
    cmap="viridis",
)


Composite soundings are useful for examining:

- temporal evolution;
- forecast-versus-observation differences;
- analog comparisons;
- model-to-model differences;
- sensitivity experiments.

You can also specify individual colors, line widths, line styles, and alpha
values with `colors_to_use`, `lw_to_use`, `ls_to_use`, and `alphas_to_use`.

**## Recipe 9 — Export a Profile**

**Goal**: Write SounderPy `clean_data` to a file for use outside the package.

## CSV

In [ ]:
spy.to_file(
    "csv",
    obs_data,
    filename="oax_20140616_18z.csv",
)


SounderPy can also export SHARPpy and CM1 formats:

```python
spy.to_file(
    "sharppy",
    obs_data,
    filename="oax_20140616_18z.snd",
)

spy.to_file(
    "cm1",
    obs_data,
    filename="input_sounding",
)
```

For CM1 export, `convert_to_AGL=True` is the default and converts height to
above-ground level.

**## Recipe 10 — Use the SounderPy Command Line**

**Goal**: See how many of the same workflows can be performed without writing a Python script.

SounderPy 3.2 includes a command-line interface.

From a terminal:

```bash
sounderpy --help
```

Retrieve the same OAX observation:

```bash
sounderpy obs OAX 2014-06-16 18
```

Create and save a sounding:

```bash
sounderpy obs OAX 2014-06-16 18 \
    --plot sounding \
    --map-zoom 0 \
    --plot-file oax_sounding.png
```

Export the profile:

```bash
sounderpy obs OAX 2014-06-16 18 \
    --output oax.csv
```

Return machine-readable JSON:

```bash
sounderpy obs OAX 2014-06-16 18 --json
```

The CLI is especially useful for shell scripts, quick checks, batch workflows,
and users who do not need a full Python notebook.

# Troubleshooting Tips

### A retrieval fails

SounderPy depends on several external archives and services. If a request that
previously worked suddenly fails, the upstream data provider may be temporarily
unavailable.

Try the request again later and check the SounderPy documentation for
source-specific availability notes.

### A plot takes longer than expected

Full SounderPy plots perform many calculations and may also request radar/map
data. For a lightweight workflow, use:

```python
spy.build_sounding(
    data,
    special_parcels="simple",
    radar=None,
    map_zoom=0,
)
```

### ECAPE or a storm-relative quantity is unavailable

Some calculations require sufficient vertical coverage, valid winds, and a
valid storm-motion solution. Sparse or incomplete profiles may not support every
diagnostic.

### A model profile differs from a nearby observation

That does not necessarily indicate an error. Model profiles are gridded,
spatially representative analyses or forecasts, while radiosondes are
observations that drift through space and time.

# 🎉 You Finished the SounderPy Beginner's Cookbook!

You now know how to:

- retrieve observed atmospheric profiles;
- understand SounderPy's `clean_data` structure;
- build and customize soundings;
- create ground-relative and storm-relative hodographs;
- retrieve RAP/RUC, BUFKIT, and ACARS data;
- calculate thermodynamic and kinematic parameters;
- compare multiple profiles;
- export SounderPy data;
- use the SounderPy command line.

## Where to go next

- **Getting Started:**  
  https://kylejgillett.github.io/sounderpy/getting_started.html

- **Tutorials:**  
  https://kylejgillett.github.io/sounderpy/tutorials/index.html

- **Plot Gallery:**  
  https://kylejgillett.github.io/sounderpy/examplegallery.html

- **Case Studies:**  
  https://kylejgillett.github.io/sounderpy/case_studies/index.html

- **Full Documentation:**  
  https://kylejgillett.github.io/sounderpy/

Thanks for using SounderPy! 🌩️